# YOLO11s + GRL using the native Ultralytics trainer

This notebook includes the updated setup for Kaggle kernels using **Python 3.13**:
training runs in an isolated **Python 3.12** environment created by `uv`.

Import this `.ipynb` into Kaggle, attach **adaptiverdd-prepared**, select a **GPU**,
enable **Internet**, and run the cells from the beginning. If an older setup cell
failed with `ensurepip`, use this notebook's environment cell instead.
The complete project is embedded, so no GitHub push is required.

Only the GRL architecture, domain objective and target batch supply are added.
Optimizer, warmup, LR scheduling, AMP, EMA, validation and source/target augmentations
follow **Ultralytics 8.3.0 defaults** unless explicitly overridden in the YAML cell.
Default batch size is 16 source images plus a target batch; uncomment `batch: 4`
if GPU memory requires it. Only one GPU is used. Source labels train detection;
target annotations are never read. Validation uses a labeled split and best
checkpoints use native validation fitness.

In [ ]:
from pathlib import Path
import json
import os
import subprocess
import sys
import zipfile

DATASET_ROOT = "/kaggle/input/datasets/doquocbinhbinh/adaptiverdd-prepared"
# Set DATASET_ROOT="" only if you want automatic discovery.
PROBE_IMAGES = ["source/images/China_Drone_000000.jpg",
                "target_adaptation/images/China_MotorBike_000001.jpg"]
# Optional direct-read diagnostics; use [] for a dataset with different filenames.
PROJECT_DIR = Path("/kaggle/working/yolo11s-grl")
NAMES = ["D00", "D10", "D20", "D40"]
TARGET_SPLIT = "target_adaptation"  # use "target" for a legacy dataset
VALIDATION_SPLIT = "target_eval"  # or a separate labeled source_eval split
TEST_SPLIT = "target_test"       # set "" to disable final test evaluation
RUN_TESTS = True
RUN_BASELINE = False
TORCH_INDEX_URL = "https://pypi.org/simple"  # avoids the timed-out download.pytorch.org host
RESUME = ""  # trusted native weights/resume.pt from an interrupted run

## 1. Restore the project
Code is refreshed on rerun; existing outputs are preserved.

In [ ]:
PROJECT_FILES = {'train.py': '"""Compatibility launcher; the installable CLI lives in grl_yolo.cli."""\nfrom grl_yolo.cli import export_checkpoints, main, make_detector, save_detector\n\n\nif __name__ == "__main__":\n    main()\n', 'requirements.txt': 'ultralytics==8.3.0\ntorch==2.5.1\ntorchvision==0.20.1\nnumpy<2\npytest>=8,<9\n', 'pyproject.toml': '[build-system]\nrequires = ["setuptools>=68"]\nbuild-backend = "setuptools.build_meta"\n\n[project]\nname = "yolo11s-grl"\nversion = "0.1.0"\ndescription = "Local multiscale GRL extension for native Ultralytics YOLO11 training"\nrequires-python = ">=3.10,<3.13"\ndependencies = ["ultralytics==8.3.0", "torch==2.5.1", "torchvision==0.20.1", "numpy<2"]\n\n[project.optional-dependencies]\ntest = ["pytest>=8,<9"]\n\n[project.scripts]\nyolo11s-grl = "grl_yolo.cli:main"\n\n[tool.setuptools.packages.find]\ninclude = ["grl_yolo*"]\n', 'pytest.ini': '[pytest]\npythonpath = .\ntestpaths = tests\n', 'README.md': '# YOLO11s + local multiscale GRL extension\n\nThe workflow is: install dependencies if needed → apply the local extension →\nload the model YAML → edit training hyperparameters → train using Ultralytics\'\nnative loop and default result folders.\n\nLocal `GRLDetectionModel` and `GRLTrainer` subclasses apply the changes at runtime.\nThe installed Ultralytics source files are not overwritten. This follows its\ncustom-trainer extension mechanism and makes the base detector independently\nusable. The implementation targets Ultralytics 8.3.0 / PyTorch 2.5.1.\n\n## 1. Install the local extension\n\nUse Python 3.10–3.12. From this folder:\n\n```bash\npython -m pip install -e .\n```\n\nThis installs missing pinned dependencies and registers the `yolo11s-grl`\ncommand. An editable install uses the files in `grl_yolo/` directly, so local\ncode edits take effect in a new Python process. For tests:\n\n```bash\npython -m pip install -e ".[test]"\npython -m pytest -q\n```\n\nInstall an appropriate PyTorch CUDA build first if your local platform needs one.\nThe Kaggle notebook creates an isolated Python 3.12 environment with PyPI Linux\nCUDA packages using uv. The notebook kernel may remain on Python 3.13.\n\n## 2. Load and edit the architecture YAML\n\n`configs/yolo11s-grl.yaml` contains the YOLO11s backbone, neck/detection head,\nsmall-model scaling and a `grl` section:\n\n```yaml\ngrl:\n  enabled: true\n  layers: [4, 6, 10]\n  hidden_channels: null\n  norm: group\n  groups: 8\n  lambda: 0.1\n  domain_weight: 1.0\n  schedule: constant\n```\n\nThe local extension registers a classifier for each tapped feature map. Each\nbranch applies GRL → Conv 1×1 → normalization → LeakyReLU → Conv 1×1.\n`hidden_channels` can specify one width per branch. GroupNorm or BatchNorm is\nselectable. Target images traverse only the backbone, so target data does not\nupdate neck/head BatchNorm statistics or receive detection loss.\n\nWhen changing backbone indices, update `grl.layers` and normal YOLO graph\nconnections accordingly. Model YAMLs must describe a valid detection graph.\nThe class count is overridden from dataset `names` by the native trainer.\n\nUse the supplied entry point or `GRLYOLO` API to interpret `grl`. Plain\n`ultralytics.YOLO(model_yaml)` parses the standard graph but does not activate\nthis extension\'s GRL branches or target-batch training.\n\n## 3. Edit dataset and training hyperparameters\n\n`configs/adaptiverdd.yaml` describes the dataset:\n\n```yaml\npath: /path/to/adaptiverdd-prepared\ntrain: source/images\ntarget: target_adaptation/images\nval: target_eval/images\ntest: target_test/images\nnames: [D00, D10, D20, D40]\n```\n\nSource/validation/test images need parallel `labels/` folders with normalized\nYOLO detection rows. Target training needs only images, and never reads target\nlabels. A labeled validation split is required. Keep final test images held out.\n\n`configs/train.yaml` selects architecture, initialization and data:\n\n```yaml\ntrain:\n  model: yolo11s-grl.yaml\n  pretrained: yolo11s.pt\n  data: adaptiverdd.yaml\n  # Optional overrides:\n  # epochs: 100\n  # batch: 4\n  # imgsz: 640\n  # device: 0\n  # lr0: 0.01\n  # optimizer: auto\n```\n\nPaths to local model/data files resolve relative to the training YAML.\n`yolo11s.pt` is downloaded by Ultralytics if absent, then matching weights are\ntransferred into the YAML-defined model. Set `pretrained: false` for random\ninitialization, or supply a compatible local checkpoint.\n\nAll omitted training settings retain Ultralytics 8.3.0 defaults, including\noptimizer, loss gains, warmup, LR scheduling, AMP, EMA, augmentation, early\nstopping, validation and output-directory naming. Source and target both use\nthe native YOLODataset augmentation pipeline, including mosaic closure.\n`batch` is the source batch size plus a target batch of the same configured size.\nThe current GRL adapter supports one GPU and a positive explicit batch size;\nnative AutoBatch does not account for target autograd graphs.\n\nYou may add a top-level `grl` section to training YAML for experiment-specific\noverrides; these take precedence over the model YAML\'s GRL settings. Existing\ntraining configs with inline GRL settings remain supported.\n\n## 4. Train\n\n```bash\nyolo11s-grl --config configs/train.yaml\n```\n\nWithout an editable installation, the compatibility launcher works:\n\n```bash\npython train.py --config configs/train.yaml\n```\n\nSource-only baseline using the same native settings:\n\n```bash\nyolo11s-grl --config configs/train.yaml --baseline\n```\n\nOr use the local model API:\n\n```python\nfrom grl_yolo.api import GRLYOLO\n\nmodel = GRLYOLO("configs/yolo11s-grl.yaml")\nmodel.load("yolo11s.pt")  # downloads if needed\nresults = model.train(data="configs/adaptiverdd.yaml", epochs=100, batch=4)\n```\n\nThe native trainer optimizes its original summed source detection loss plus\n`source_batch_size * domain_weight * mean_domain_BCE`.\nDomain BCE averages source=1/target=0 and then scales. GRL multiplies feature\ngradients by `-lambda`, while classifier gradients keep their ordinary sign.\n\n## 5. Native results and resume\n\nWith `project` and `name` omitted, native training results go to\n`runs/detect/train`, `train2`, etc. relative to the process working directory.\nSet standard `train.project`/`train.name` only when you want an explicit override.\nBaseline runs use the same native directory incrementing. Notebook validation\nand prediction also use native save directories.\n\nNative outputs include `weights/best.pt`, `weights/last.pt`, `results.csv`,\n`args.yaml`, metrics and plots. Best weights use native validation fitness.\nAdditional GRL outputs in the same run directory are `grl.yaml`, `domain.csv`,\n`configuration.yaml` and `weights/resume.pt`. The CLI also exports standard\n`weights/best_detector.pt` and `weights/last_detector.pt` without custom modules.\nThe API\'s native best/last checkpoints contain the local model class and require\nthis package to remain importable; use `export_checkpoints(model.trainer)` from\n`grl_yolo.cli` for standard detector exports after an API run.\n\n```bash\nyolo11s-grl --config configs/train.yaml --resume runs/detect/train/weights/resume.pt\nyolo detect val model=runs/detect/train/weights/best_detector.pt data=configs/adaptiverdd.yaml split=test\n```\n\nResume an interrupted run using the same GRL settings. Native resume restores\nthe original epoch budget. For additional training after a completed run, start\na fresh run initialized from its detector weights. Checkpoints from the older\nstandalone loop (`adaptation.pt`) cannot resume this native trainer.\n\n## Kaggle notebook\n\nFor the GitHub workflow, upload the contents of this project folder to the root\nof [GRL_RDD26](https://github.com/binhdoquoc/GRL_RDD26) (so `train.py`, `grl_yolo/`\nand `configs/` are directly at the repository root). Do not upload datasets,\nenvironments, model weights or run outputs. Then import\n`kaggle_yolo11s_grl_clone.ipynb` into Kaggle. Its `REPO_URL` is configured for that\nrepository. Set\noptionally `REPO_REF` (branch, tag or commit), attach the dataset, enable GPU and\nInternet, and run in order. An empty ref clones the repository default branch.\nThe notebook prints the checked-out commit, uses the local extension from that\ncheckout, and reuses it on reruns without pulling over edited configuration.\nUse a fresh session to fetch an updated version. Results remain in the project\'s\nnative `runs/detect` folders and are packaged into the downloadable results ZIP.\n\nImport `kaggle_yolo11s_grl.ipynb`, attach the dataset, enable GPU and Internet,\nand run in order. It restores local extension files, installs pinned dependencies\nif needed, loads `configs/yolo11s-grl.yaml`, writes editable training YAML, trains,\nevaluates and packages the actual native run directories. No GitHub push is needed.\n\nIt reads the supplied dataset paths directly and uses no symlink staging or\ncustom directory-access gate. Optional file probes only print diagnostics.\nThe native loader still needs readable source/target images in the running session.\nOn read-only Kaggle inputs, an unsaved label-cache warning does not require copies.\n\nSetup avoids ensurepip through uv and uses PyPI by default with custom-index\nfallback. It forces Matplotlib `Agg` for subprocesses and displays saved plots\nin the kernel. Training output is streamed and saved to `logs/train.log`.\n\nRegenerate both notebooks after notebook workflow changes (and the embedded\nversion after local code/config changes):\n\n```bash\npython scripts/create_kaggle_notebook.py\n```\n\n## Verification\n\nTests cover analytical GRL/domain gradients, real YOLO11 backward passes, native\ntraining/resume, YAML architecture and experiment override precedence, pretrained\ntransfer, default output directories, standard exports, bootstrap/logging and\nnotebook packaging. Verification uses synthetic images on CPU; AdaptiveRDD\naccuracy and live Kaggle GPU execution remain unmeasured.\n\nReferences: [Ultralytics customization](https://docs.ultralytics.com/guides/custom-trainer),\n[model YAML guide](https://docs.ultralytics.com/guides/model-yaml-config),\n[MS-DAYOLO](https://github.com/Mazin-Hnewa/MS-DAYOLO).\nUltralytics and the adapted model YAML use AGPL-3.0 licensing.\n', 'configs/train.yaml': '# Architecture/GRL: model YAML. Dataset: data YAML.\n# Omitted keys, including project/name, inherit Ultralytics defaults.\ntrain:\n  model: yolo11s-grl.yaml\n  pretrained: yolo11s.pt  # automatically downloaded if absent\n  data: adaptiverdd.yaml\n  # Optional experiment overrides:\n  # epochs: 100\n  # batch: 4\n  # imgsz: 640\n  # device: 0\n  # workers: 2\n  # optimizer: auto\n  # lr0: 0.01\n  # weight_decay: 0.0005\n  # mosaic: 1.0\n  # mixup: 0.0\n  # close_mosaic: 10\n  # project: ../runs/custom\n  # name: experiment\n\n# Optional GRL overrides per experiment:\n# grl:\n#   lambda: 0.2\n#   domain_weight: 0.5\n', 'configs/yolo11s-grl.yaml': '# Adapted from Ultralytics v8.3.0 YOLO11 (AGPL-3.0).\n# GRLDetectionModel reads grl and registers the training-only branches.\nnc: 80\nscale: s\nscales:\n  s: [0.50, 0.50, 1024]\nbackbone:\n  - [-1, 1, Conv, [64, 3, 2]]\n  - [-1, 1, Conv, [128, 3, 2]]\n  - [-1, 2, C3k2, [256, False, 0.25]]\n  - [-1, 1, Conv, [256, 3, 2]]\n  - [-1, 2, C3k2, [512, False, 0.25]]\n  - [-1, 1, Conv, [512, 3, 2]]\n  - [-1, 2, C3k2, [512, True]]\n  - [-1, 1, Conv, [1024, 3, 2]]\n  - [-1, 2, C3k2, [1024, True]]\n  - [-1, 1, SPPF, [1024, 5]]\n  - [-1, 2, C2PSA, [1024]]\nhead:\n  - [-1, 1, nn.Upsample, [null, 2, nearest]]\n  - [[-1, 6], 1, Concat, [1]]\n  - [-1, 2, C3k2, [512, False]]\n  - [-1, 1, nn.Upsample, [null, 2, nearest]]\n  - [[-1, 4], 1, Concat, [1]]\n  - [-1, 2, C3k2, [256, False]]\n  - [-1, 1, Conv, [256, 3, 2]]\n  - [[-1, 13], 1, Concat, [1]]\n  - [-1, 2, C3k2, [512, False]]\n  - [-1, 1, Conv, [512, 3, 2]]\n  - [[-1, 10], 1, Concat, [1]]\n  - [-1, 2, C3k2, [1024, True]]\n  - [[16, 19, 22], 1, Detect, [nc]]\ngrl:\n  enabled: true\n  layers: [4, 6, 10]\n  hidden_channels: null\n  norm: group\n  groups: 8\n  lambda: 0.1\n  domain_weight: 1.0\n  schedule: constant\n', 'configs/adaptiverdd.yaml': "# Replace this root with your dataset location. Paths below are relative to it.\npath: /kaggle/input/datasets/doquocbinhbinh/adaptiverdd-prepared\ntrain: source/images\nval: target_eval/images\ntest: target_test/images\n# Used only by this project's adaptation loader; labels are never read.\ntarget: target_adaptation/images\nnames:\n  0: D00\n  1: D10\n  2: D20\n  3: D40\n", 'grl_yolo/__init__.py': '"""Multiscale gradient reversal domain adaptation for YOLO11s."""\n', 'grl_yolo/adaptation.py': '"""Training-only spatial domain classifiers; detector inference stays standard."""\nimport torch\nfrom torch import nn\nfrom torch.nn import functional as F\n\n\nclass _Reverse(torch.autograd.Function):\n    @staticmethod\n    def forward(ctx, x, coefficient):\n        ctx.coefficient = coefficient\n        return x.view_as(x)\n\n    @staticmethod\n    def backward(ctx, gradient):\n        return -ctx.coefficient * gradient, None\n\n\ndef reverse_gradient(x, coefficient=0.1):\n    return _Reverse.apply(x, coefficient)\n\n\ndef capture_features(model, inputs, layers=(4, 6, 10)):\n    """Capture backbone P3/P4/P5 without persistent hooks in saved models."""\n    features = {}\n\n    def hook(index):\n        def capture(module, args, output):\n            features[index] = output\n        return capture\n\n    handles = [model.model[i].register_forward_hook(hook(i)) for i in layers]\n    try:\n        result = model(inputs)\n    finally:\n        for handle in handles:\n            handle.remove()\n    return result, [features[i] for i in layers]\n\n\nclass MultiscaleDomainHeads(nn.Module):\n    def __init__(self, channels, hidden_channels=None, norm="group", groups=8):\n        super().__init__()\n        hidden_channels = hidden_channels or [max(32, c // 2) for c in channels]\n        if len(hidden_channels) != len(channels):\n            raise ValueError("hidden_channels must contain one width per GRL feature layer")\n        self.heads = nn.ModuleList()\n        for c, hidden in zip(channels, hidden_channels):\n            if norm == "group":\n                normalization = nn.GroupNorm(groups, hidden)\n            elif norm == "batch":\n                normalization = nn.BatchNorm2d(hidden)\n            else:\n                raise ValueError("GRL norm must be \'group\' or \'batch\'")\n            self.heads.append(nn.Sequential(nn.Conv2d(c, hidden, 1), normalization,\n                                            nn.LeakyReLU(0.1), nn.Conv2d(hidden, 1, 1)))\n\n    def forward(self, source, target, coefficient=0.1):\n        if len(source) != len(self.heads) or len(target) != len(self.heads):\n            raise ValueError("One source and target feature map is required per scale")\n        losses = []\n        for head, src, tgt in zip(self.heads, source, target):\n            # Match original domain convention: source=1, target=0.\n            s = head(reverse_gradient(src, coefficient))\n            t = head(reverse_gradient(tgt, coefficient))\n            losses.append(0.5 * (\n                F.binary_cross_entropy_with_logits(s, torch.ones_like(s))\n                + F.binary_cross_entropy_with_logits(t, torch.zeros_like(t))\n            ))\n        return torch.stack(losses).mean()\n', 'grl_yolo/cli.py': '"""Native Ultralytics training with YAML-configured GRL architecture additions."""\nimport argparse\nimport json\nfrom copy import deepcopy\nfrom pathlib import Path\n\nimport torch\nimport ultralytics\nfrom ultralytics import YOLO\nfrom ultralytics.models.yolo.detect import DetectionTrainer\nfrom ultralytics.nn.tasks import DetectionModel, attempt_load_one_weight\nfrom ultralytics.utils import yaml_save\n\nfrom grl_yolo.config import load_training_config\nfrom grl_yolo.model import plain_detector\nfrom grl_yolo.trainer import GRLTrainer\n\n\ndef make_detector(names, weights="yolo11s.pt"):\n    """Build an ordinary detector for regression tests and compatible weights."""\n    base = YOLO("yolo11s.yaml").model\n    model = DetectionModel(deepcopy(base.yaml), nc=len(names), verbose=False)\n    if weights:\n        model.load(YOLO(weights).model)\n    model.names = names\n    for name, parameter in model.named_parameters():\n        parameter.requires_grad_(".dfl." not in name)\n    return model\n\n\ndef save_detector(model, filename, args):\n    detector = plain_detector(model)\n    detector.args = deepcopy(args)\n    torch.save({"model": detector, "train_args": vars(args) if not isinstance(args, dict) else args,\n                "version": ultralytics.__version__}, filename)\n\n\ndef export_checkpoints(trainer):\n    for name in ("last", "best"):\n        checkpoint = trainer.wdir / f"{name}.pt"\n        if checkpoint.exists():\n            model, _ = attempt_load_one_weight(checkpoint, device=torch.device("cpu"))\n            save_detector(model, trainer.wdir / f"{name}_detector.pt", model.args)\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=__doc__)\n    parser.add_argument("--config", default="configs/train.yaml")\n    parser.add_argument("--report", help="Write the actual run directory to JSON for notebook use")\n    for key in ("data", "model", "device", "name", "project", "resume"):\n        parser.add_argument(f"--{key}")\n    for key in ("epochs", "batch", "imgsz", "workers"):\n        parser.add_argument(f"--{key}", type=int)\n    parser.add_argument("--baseline", action="store_true", help="Use the unmodified DetectionTrainer")\n    cli = parser.parse_args()\n    if ultralytics.__version__ != "8.3.0":\n        raise RuntimeError("Install requirements.txt: this adapter targets ultralytics==8.3.0")\n    overrides, grl = load_training_config(cli.config)\n    for key, value in vars(cli).items():\n        if key not in {"config", "baseline", "report"} and value is not None:\n            overrides[key] = value\n    if cli.baseline:\n        grl["enabled"] = False\n    if grl["enabled"] and grl["domain_weight"] > 0:\n        trainer = GRLTrainer(overrides=overrides, grl=grl)\n    else:\n        trainer = DetectionTrainer(overrides=overrides)\n    yaml_save(trainer.save_dir / "configuration.yaml", {"train": overrides, "grl": grl})\n    trainer.train()\n    export_checkpoints(trainer)\n    if cli.report:\n        Path(cli.report).write_text(json.dumps({"run_dir": str(trainer.save_dir.resolve())}), encoding="utf-8")\n    print(f"Results: {trainer.save_dir}")\n\n\nif __name__ == "__main__":\n    main()\n', 'grl_yolo/api.py': '"""Apply local GRL changes using the standard Ultralytics model/trainer API."""\nfrom ultralytics import YOLO\nfrom ultralytics.models.yolo.detect import DetectionTrainer\n\nfrom .model import GRLDetectionModel\nfrom .trainer import GRLTrainer\n\n\nclass GRLYOLO(YOLO):\n    @property\n    def task_map(self):\n        mapping = super().task_map\n        mapping["detect"] = {**mapping["detect"], "model": GRLDetectionModel, "trainer": GRLTrainer}\n        return mapping\n\n    def train(self, trainer=None, **kwargs):\n        settings = getattr(self.model, "grl_config", {})\n        if trainer is None and (not settings.get("enabled", True) or settings.get("domain_weight", 1.0) == 0):\n            trainer = DetectionTrainer\n        return super().train(trainer=trainer, **kwargs)\n', 'grl_yolo/config.py': '"""Separate GRL-only settings from unchanged Ultralytics training settings."""\nfrom copy import deepcopy\nfrom pathlib import Path\n\nimport yaml\n\n\nGRL_DEFAULTS = {"enabled": True, "layers": [4, 6, 10], "lambda": 0.1,\n                "domain_weight": 1.0, "schedule": "constant", "hidden_channels": None,\n                "norm": "group", "groups": 8}\n\n\ndef grl_settings(values=None):\n    values = values or {}\n    unknown = set(values) - set(GRL_DEFAULTS)\n    if unknown:\n        raise ValueError(f"Unknown GRL settings: {sorted(unknown)}")\n    settings = {**deepcopy(GRL_DEFAULTS), **values}\n    if settings["lambda"] < 0 or settings["domain_weight"] < 0:\n        raise ValueError("GRL lambda and domain_weight must be nonnegative")\n    if settings["schedule"] not in {"constant", "dann"}:\n        raise ValueError("GRL schedule must be constant or dann")\n    layers = settings["layers"]\n    if not layers or len(set(layers)) != len(layers) or any(not isinstance(i, int) or i < 0 for i in layers):\n        raise ValueError("GRL layers must be distinct nonnegative backbone indices")\n    return settings\n\n\ndef load_training_config(filename):\n    filename = Path(filename).resolve()\n    raw = yaml.safe_load(filename.read_text(encoding="utf-8")) or {}\n    if set(raw) - {"train", "grl"}:\n        raise ValueError("Training YAML must contain only train and grl sections")\n    train = deepcopy(raw.get("train", {}))\n    # Only supplied settings are overrides; all omitted keys come from Ultralytics.\n    for key in ("data", "project"):\n        if key in train and train[key] is not None:\n            path = Path(train[key])\n            train[key] = str(path if path.is_absolute() else (filename.parent / path).resolve())\n    for key in ("model", "resume", "pretrained"):\n        value = train.get(key)\n        if isinstance(value, str):\n            path = filename.parent / value\n            if path.exists():\n                train[key] = str(path.resolve())\n    architecture_grl = {}\n    model = train.get("model")\n    if isinstance(model, str) and Path(model).suffix.lower() in {".yaml", ".yml"} and Path(model).is_file():\n        architecture = yaml.safe_load(Path(model).read_text(encoding="utf-8")) or {}\n        architecture_grl = architecture.get("grl", {})\n    return train, grl_settings({**architecture_grl, **(raw.get("grl") or {})})\n', 'grl_yolo/model.py': '"""YOLO11 detection architecture plus registered training-only GRL branches."""\nfrom copy import deepcopy\n\nimport torch\nfrom ultralytics.nn.tasks import DetectionModel\n\nfrom .adaptation import MultiscaleDomainHeads, capture_features\nfrom .config import grl_settings\n\n\nclass GRLDetectionModel(DetectionModel):\n    def __init__(self, cfg="yolo11s.yaml", nc=None, grl=None, verbose=True):\n        super().__init__(cfg=cfg, nc=nc, verbose=verbose)\n        self.grl_config = grl_settings({**self.yaml.get("grl", {}), **(grl or {})})\n        layers = self.grl_config["layers"]\n        if max(layers) >= len(self.yaml["backbone"]):\n            raise ValueError("GRL feature layers must belong to the YOLO11 backbone")\n        self.grl_coefficient = self.grl_config["lambda"]\n        was_training = self.training\n        self.eval()\n        with torch.no_grad():\n            features = self.backbone_features(torch.zeros(1, 3, 128, 128))\n        self.train(was_training)\n        self.domain_heads = MultiscaleDomainHeads(\n            [f.shape[1] for f in features], self.grl_config["hidden_channels"],\n            self.grl_config["norm"], self.grl_config["groups"],\n        )\n        self.last_domain_loss = 0.0\n        self.last_total_mean = 0.0\n\n    def backbone_features(self, images):\n        """Target images stop at the backbone; the neck/head remain source-only."""\n        layers = self.grl_config["layers"]\n        features, saved = {}, []\n        x = images\n        for index, module in enumerate(self.model):\n            if module.f != -1:\n                x = saved[module.f] if isinstance(module.f, int) else [\n                    x if source == -1 else saved[source] for source in module.f\n                ]\n            x = module(x)\n            saved.append(x if module.i in self.save else None)\n            if index in layers:\n                features[index] = x\n            if index == max(layers):\n                break\n        return [features[index] for index in layers]\n\n    def loss(self, batch, preds=None):\n        # Standard source-only loss is also used by the Ultralytics validator.\n        if not self.training or "target_img" not in batch:\n            return super().loss(batch, preds)\n        predictions, source = capture_features(self, batch["img"], self.grl_config["layers"])\n        detection_loss, items = super().loss(batch, predictions)\n        target = self.backbone_features(batch["target_img"])\n        domain_loss = self.domain_heads(source, target, self.grl_coefficient)\n        # Keep Ultralytics\' summed-over-images detection loss exactly unchanged.\n        batch_size = batch["img"].shape[0]\n        total = detection_loss + batch_size * self.grl_config["domain_weight"] * domain_loss\n        self.last_domain_loss = float(domain_loss.detach())\n        self.last_total_mean = float(total.detach()) / batch_size\n        return total, items\n\n\ndef plain_detector(model):\n    """Export the detector as a standard Ultralytics class, without custom modules."""\n    architecture = deepcopy(model.yaml)\n    architecture.pop("grl", None)\n    detector = DetectionModel(architecture, nc=model.model[-1].nc, verbose=False)\n    detector.load(model, verbose=False)\n    detector.names = deepcopy(model.names)\n    detector.args = deepcopy(model.args)\n    return detector.cpu().float().eval()\n', 'grl_yolo/trainer.py': '"""Inherit the native loop; customize only GRL architecture and target batches."""\nimport csv\nimport math\nimport shutil\nfrom copy import deepcopy\nfrom pathlib import Path\n\nfrom ultralytics.data import build_dataloader\nfrom ultralytics.models.yolo.detect import DetectionTrainer\nfrom ultralytics.nn.tasks import yaml_model_load\nfrom ultralytics.utils import DEFAULT_CFG, yaml_save\nfrom ultralytics.utils.torch_utils import de_parallel\n\nfrom .config import grl_settings\nfrom .data import UnlabeledYOLODataset\nfrom .model import GRLDetectionModel\n\n\nclass GRLTrainer(DetectionTrainer):\n    def __init__(self, cfg=DEFAULT_CFG, overrides=None, _callbacks=None, grl=None):\n        self.grl_overrides = grl\n        self.grl = grl_settings(grl)\n        self.target_loader = self.target_iterator = None\n        self.domain_rows = []\n        self.domain_step = 0\n        super().__init__(cfg=cfg, overrides=overrides, _callbacks=_callbacks)\n        device = self.args.device\n        if (isinstance(device, (list, tuple)) and len(device) > 1) or (isinstance(device, str) and "," in device):\n            raise ValueError("GRL currently supports one GPU; use device: 0")\n        if self.args.batch < 1:\n            raise ValueError("Specify a positive batch size; native AutoBatch excludes target graphs")\n        self.add_callback("on_train_epoch_start", self._start_domain_epoch)\n        self.add_callback("on_train_batch_start", self._set_coefficient)\n        self.add_callback("on_train_batch_end", self._record_domain_batch)\n        self.add_callback("on_train_epoch_end", self._write_domain_epoch)\n\n    def get_model(self, cfg=None, weights=None, verbose=True):\n        architecture = cfg if isinstance(cfg, dict) else yaml_model_load(cfg)\n        inherited = architecture.get("grl", {})\n        if self.resume and isinstance(weights, GRLDetectionModel):\n            inherited = weights.grl_config\n        self.grl = grl_settings({**inherited, **(self.grl_overrides or {})})\n        if isinstance(weights, GRLDetectionModel) and self.resume and weights.grl_config != self.grl:\n            raise ValueError("GRL YAML differs from the resume checkpoint")\n        yaml_save(self.save_dir / "grl.yaml", self.grl)\n        model = GRLDetectionModel(architecture, nc=self.data["nc"], grl=self.grl, verbose=verbose)\n        if weights:\n            model.load(weights)\n        return model\n\n    def get_dataloader(self, dataset_path, batch_size=16, rank=0, mode="train"):\n        loader = super().get_dataloader(dataset_path, batch_size, rank, mode)\n        if mode == "train":\n            target = self.data.get("target")\n            if not target:\n                raise ValueError("Dataset YAML needs target: for unlabeled training images")\n            root = Path(self.data["path"])\n            def absolute(p):\n                return str(Path(p) if Path(p).is_absolute() else root / p)\n            target = absolute(target) if isinstance(target, str) else [absolute(p) for p in target]\n            dataset = UnlabeledYOLODataset(\n                img_path=target, imgsz=self.args.imgsz, batch_size=batch_size, augment=True,\n                hyp=deepcopy(self.args), rect=self.args.rect, cache=self.args.cache or None,\n                single_cls=self.args.single_cls, stride=self.stride, pad=0.0, prefix="target: ",\n                task="detect", classes=self.args.classes, data=self.data, fraction=self.args.fraction,\n            )\n            self.target_loader = build_dataloader(dataset, batch_size, self.args.workers,\n                                                  shuffle=not dataset.rect, rank=-1)\n            self.target_iterator = iter(self.target_loader)\n        return loader\n\n    def preprocess_batch(self, batch):\n        batch = super().preprocess_batch(batch)\n        try:\n            target = next(self.target_iterator)\n        except StopIteration:\n            self.target_iterator = iter(self.target_loader)\n            target = next(self.target_iterator)\n        batch["target_img"] = super().preprocess_batch(target)["img"]\n        return batch\n\n    def _close_dataloader_mosaic(self):\n        super()._close_dataloader_mosaic()\n        if self.target_loader:\n            self.target_loader.dataset.close_mosaic(hyp=deepcopy(self.args))\n            self.target_loader.reset()\n            self.target_iterator = iter(self.target_loader)\n\n    def _start_domain_epoch(self, trainer):\n        self.domain_rows = []\n        self.domain_step = 0\n\n    def _set_coefficient(self, trainer):\n        value = self.grl["lambda"]\n        if self.grl["schedule"] == "dann":\n            batches = len(self.train_loader)\n            progress = (self.epoch * batches + self.domain_step) / max(self.epochs * batches - 1, 1)\n            value *= 2 / (1 + math.exp(-10 * progress)) - 1\n        de_parallel(self.model).grl_coefficient = value\n\n    def _record_domain_batch(self, trainer):\n        model = de_parallel(self.model)\n        self.domain_rows.append((model.last_domain_loss, model.last_total_mean, model.grl_coefficient))\n        self.domain_step += 1\n\n    def _write_domain_epoch(self, trainer):\n        if not self.domain_rows:\n            return\n        filename = self.save_dir / "domain.csv"\n        empty = not filename.exists()\n        means = [sum(row[i] for row in self.domain_rows) / len(self.domain_rows) for i in range(3)]\n        with filename.open("a", newline="", encoding="utf-8") as stream:\n            writer = csv.writer(stream)\n            if empty:\n                writer.writerow(["epoch", "domain_loss", "total_loss_per_source_image", "grl_lambda"])\n            writer.writerow([self.epoch + 1, *means])\n\n    def save_model(self):\n        super().save_model()\n        # Preserve the native optimizer before final_eval strips last/best checkpoints.\n        shutil.copy2(self.last, self.wdir / "resume.pt")\n', 'grl_yolo/data.py': '"""Source YOLO labels and a separate, strictly image-only target loader."""\nfrom pathlib import Path\n\nimport cv2\nimport numpy as np\nimport torch\nimport yaml\nfrom torch.utils.data import Dataset\nfrom ultralytics.data.augment import LetterBox\nfrom ultralytics.data.utils import IMG_FORMATS\nfrom ultralytics.data.dataset import YOLODataset\nfrom PIL import Image\n\n\nclass UnlabeledYOLODataset(YOLODataset):\n    """Use the native augmentation pipeline without reading target annotations."""\n    def get_labels(self):\n        labels = []\n        for filename in self.im_files:\n            with Image.open(filename) as image:\n                width, height = image.size\n            labels.append({"im_file": filename, "shape": (height, width),\n                           "cls": np.zeros((0, 1), dtype=np.float32),\n                           "bboxes": np.zeros((0, 4), dtype=np.float32),\n                           "segments": [], "keypoints": None,\n                           "normalized": True, "bbox_format": "xywh"})\n        return labels\n\n\ndef load_config(filename):\n    filename = Path(filename).resolve()\n    data = yaml.safe_load(filename.read_text(encoding="utf-8"))\n    root = Path(data.get("path", filename.parent))\n    if not root.is_absolute():\n        root = filename.parent / root\n    root = root.resolve()\n    for key in ("train", "val", "test", "target"):\n        if key in data:\n            if not isinstance(data[key], str):\n                raise ValueError(f"{key} must be one image directory or .txt manifest")\n            value = Path(data[key])\n            data[key] = str(value if value.is_absolute() else root / value)\n    names = data["names"]\n    data["names"] = dict(enumerate(names)) if isinstance(names, list) else names\n    if set(data["names"]) != set(range(len(data["names"]))):\n        raise ValueError("Class names must use consecutive integer IDs starting at 0")\n    data["nc"] = len(data["names"])\n    data["path"] = str(root)\n    return data\n\n\nclass TargetImages(Dataset):\n    def __init__(self, location, imgsz):\n        path = Path(location).resolve()\n        if path.is_dir():\n            paths = sorted(path.rglob("*"))\n        elif path.is_file() and path.suffix.lower() == ".txt":\n            paths = []\n            for line in path.read_text(encoding="utf-8").splitlines():\n                if line.strip():\n                    item = Path(line.strip())\n                    paths.append(item if item.is_absolute() else path.parent / item)\n        else:\n            raise FileNotFoundError(f"Target directory or manifest not found: {path}")\n        self.paths = [p for p in paths if p.suffix.lower().lstrip(".") in IMG_FORMATS]\n        if not self.paths:\n            raise ValueError(f"No target images found in {path}")\n        missing = [p for p in self.paths if not p.is_file()]\n        if missing:\n            raise FileNotFoundError(f"Target image not found: {missing[0]}")\n        self.letterbox = LetterBox((imgsz, imgsz), auto=False, scaleup=True)\n\n    def __len__(self):\n        return len(self.paths)\n\n    def __getitem__(self, index):\n        image = cv2.imread(str(self.paths[index]))\n        if image is None:\n            raise ValueError(f"Cannot decode target image: {self.paths[index]}")\n        image = self.letterbox(image=image)\n        image = np.ascontiguousarray(image[:, :, ::-1].transpose(2, 0, 1))\n        return torch.from_numpy(image).float() / 255.0\n', 'grl_yolo/kaggle.py': '"""Standard-library helpers for discovering and staging a Kaggle dataset."""\nimport os\nfrom pathlib import Path\n\n\ndef named_directory(parent, name):\n    """Resolve dataset directory spelling while preserving its real Linux path."""\n    parent = Path(parent)\n    direct = parent / name\n    # Explicit accessible paths need no parent-directory enumeration.\n    if direct.is_dir() or not parent.is_dir():\n        return direct\n    matches = [p for p in parent.iterdir() if p.is_dir() and p.name.casefold() == name.casefold()]\n    exact = [p for p in matches if p.name == name]\n    if exact:\n        return exact[0]\n    if len(matches) > 1:\n        raise ValueError(f"Multiple case variants for {name} under {parent}: {matches}")\n    return matches[0] if matches else direct\n\n\ndef split_images(root, split):\n    return named_directory(named_directory(root, split), "images")\n\n\ndef adaptation_split(root):\n    """Prefer the prepared dataset\'s explicit adaptation split; retain legacy support."""\n    root = Path(root)\n    for name in ("target_adaptation", "target"):\n        folder = named_directory(root, name)\n        if named_directory(folder, "images").is_dir():\n            return folder.name\n    raise FileNotFoundError(f"Expected target_adaptation/images or target/images under {root}")\n\n\ndef missing_training_paths(root):\n    """Describe incomplete mounts instead of treating them as undiscovered datasets."""\n    root = Path(root)\n    missing = []\n    source = split_images(root, "source")\n    if not source.is_dir():\n        missing.append(str(source))\n    try:\n        adaptation_split(root)\n    except FileNotFoundError:\n        name = "target_adaptation" if named_directory(root, "target_adaptation").is_dir() else "target"\n        missing.append(str(split_images(root, name)))\n    return missing\n\n\ndef discover_dataset(input_root="/kaggle/input", override=""):\n    def valid(root):\n        if not split_images(root, "source").is_dir():\n            return False\n        try:\n            adaptation_split(root)\n        except FileNotFoundError:\n            return False\n        return True\n\n    if override:\n        root = Path(override).resolve()\n        if not valid(root):\n            raise FileNotFoundError("Training dataset is incomplete. Missing image directories: "\n                                    + ", ".join(missing_training_paths(root)))\n        return root\n    input_root = Path(input_root)\n    candidates, incomplete, visited = set(), {}, set()\n    # Follow Kaggle input links, but never revisit resolved directories or walk images.\n    for current, directories, _ in os.walk(input_root, followlinks=True):\n        root = Path(current).resolve()\n        if root in visited:\n            directories[:] = []\n            continue\n        visited.add(root)\n        lower_names = {name.casefold() for name in directories}\n        if lower_names & {"source", "target_adaptation", "target"}:\n            if valid(root):\n                candidates.add(root)\n            else:\n                incomplete[root] = missing_training_paths(root)\n        directories[:] = [name for name in directories if name.casefold() not in {"images", "labels"}]\n    candidates = sorted(candidates)\n    if not candidates and incomplete:\n        details = "\\n".join(f"{root}: missing {\', \'.join(paths)}" for root, paths in incomplete.items())\n        raise FileNotFoundError("Attached training dataset is incomplete:\\n" + details +\n                                "\\nAttach source images with YOLO labels and target-adaptation images. "\n                                "Target evaluation/test images cannot substitute for missing training data.")\n    if len(candidates) != 1:\n        raise ValueError(f"Found {len(candidates)} matching datasets: {candidates}. "\n                         "Expected source/images plus target_adaptation/images (or target/images). "\n                         "Set DATASET_ROOT explicitly if multiple datasets are attached.")\n    return candidates[0]\n\n\ndef stage_dataset(root, destination):\n    """Link input images/labels; keep Ultralytics label caches in writable storage."""\n    root, destination = Path(root).resolve(), Path(destination).resolve()\n    target_split = adaptation_split(root)\n    for split in ("source", "target", "source_eval", "target_eval", "source_test", "target_test"):\n        # Keep generated training YAML stable: staged target/ maps to target_adaptation/.\n        original = named_directory(root, target_split if split == "target" else split)\n        if not named_directory(original, "images").is_dir():\n            continue\n        staged = destination / split\n        staged.mkdir(parents=True, exist_ok=True)\n        for kind in ("images", "labels"):\n            source, link = named_directory(original, kind), staged / kind\n            if not source.is_dir():\n                continue\n            if link.is_symlink():\n                if link.resolve() != source.resolve():\n                    raise FileExistsError(f"Existing link points to another dataset: {link}")\n            elif link.exists():\n                raise FileExistsError(f"Expected a dataset symlink, found an existing path: {link}")\n            else:\n                link.symlink_to(source, target_is_directory=True)\n    return destination\n\n\ndef describe_split(root, split, require_labels=False):\n    root = Path(root)\n    formats = {"jpg", "jpeg", "png", "bmp", "tif", "tiff", "webp"}\n    image_root = split_images(root, split)\n    label_root = named_directory(named_directory(root, split), "labels")\n    images = sorted(p for p in image_root.rglob("*")\n                    if p.is_file() and p.suffix.lower().lstrip(".") in formats)\n    if not images:\n        raise ValueError(f"No images in {root / split / \'images\'}")\n    if not require_labels:\n        return {"split": split, "images": len(images), "labels": "not read"}\n    labels = [label_root / p.relative_to(image_root).with_suffix(".txt") for p in images]\n    existing = sum(p.is_file() for p in labels)\n    nonempty = sum(p.is_file() and bool(p.read_text(encoding="utf-8").strip()) for p in labels)\n    if not nonempty:\n        raise ValueError(f"No nonempty YOLO labels found for {split}")\n    return {"split": split, "images": len(images), "labels": existing, "nonempty_labels": nonempty}\n', 'grl_yolo/environment.py': '"""Bootstrap Kaggle training without kernel pip, ensurepip or Python-version coupling."""\nimport json\nimport os\nimport platform\nimport shutil\nimport subprocess\nimport tarfile\nimport urllib.request\nimport uuid\nfrom pathlib import Path\n\n\nPYPI_INDEX = "https://pypi.org/simple"\n\n\ndef install_training_packages(uv, python, project, env, torch_index=PYPI_INDEX):\n    """Install pinned packages via PyPI; fall back there if a custom index fails."""\n    indexes = list(dict.fromkeys([torch_index or PYPI_INDEX, PYPI_INDEX]))\n    base = [str(uv), "pip", "install", "--python", str(python)]\n    for index, url in enumerate(indexes):\n        print("Installing pinned PyTorch from:", url, flush=True)\n        try:\n            subprocess.run([*base, "torch==2.5.1", "torchvision==0.20.1", "--index-url", url],\n                           env=env, check=True)\n            break\n        except subprocess.CalledProcessError as error:\n            if index + 1 == len(indexes):\n                raise RuntimeError(\n                    "PyTorch installation failed from PyPI. Check Kaggle Internet access and the "\n                    "installer output above. Training cannot start until the pinned packages are installed."\n                ) from error\n            print(f"Index {url} failed; retrying from PyPI.", flush=True)\n    subprocess.run([*base, "-r", str(Path(project) / "requirements.txt"), "--index-url", PYPI_INDEX],\n                   env=env, check=True)\n\n\ndef find_uv(directory):\n    """Use an installed uv or download its official Linux standalone executable."""\n    installed = shutil.which("uv")\n    if installed:\n        return Path(installed)\n    directory = Path(directory)\n    binary = directory / "uv"\n    if binary.is_file():\n        return binary\n    if platform.system() != "Linux":\n        raise RuntimeError("This bootstrap targets Kaggle Linux. Install uv locally to use it elsewhere.")\n    architecture = {"x86_64": "x86_64", "amd64": "x86_64", "aarch64": "aarch64"}.get(platform.machine().lower())\n    if not architecture:\n        raise RuntimeError(f"Unsupported Linux architecture: {platform.machine()}")\n    directory.mkdir(parents=True, exist_ok=True)\n    archive = directory / "uv.tar.gz"\n    url = f"https://github.com/astral-sh/uv/releases/latest/download/uv-{architecture}-unknown-linux-gnu.tar.gz"\n    print("Downloading standalone uv:", url, flush=True)\n    # Copy only the named regular executable; do not extract arbitrary archive paths.\n    request = urllib.request.Request(url, headers={"User-Agent": "yolo11s-grl-notebook"})\n    with urllib.request.urlopen(request, timeout=120) as response, archive.open("wb") as output:\n        shutil.copyfileobj(response, output)\n    with tarfile.open(archive, "r:gz") as bundle:\n        members = [member for member in bundle.getmembers() if member.isfile() and Path(member.name).name == "uv"]\n        if len(members) != 1:\n            raise RuntimeError("Official uv archive did not contain one regular uv executable")\n        pending = directory / "uv.pending"\n        with bundle.extractfile(members[0]) as source, pending.open("wb") as output:\n            shutil.copyfileobj(source, output)\n    pending.chmod(0o755)\n    pending.replace(binary)\n    return binary\n\n\ndef environment_python(directory):\n    return Path(directory) / ("Scripts/python.exe" if os.name == "nt" else "bin/python")\n\n\ndef python_version(python, env):\n    if not Path(python).is_file():\n        return None\n    try:\n        result = subprocess.run(\n            [str(python), "-c", "import json, sys; print(json.dumps(list(sys.version_info[:2])))"],\n            env=env, text=True, capture_output=True,\n        )\n    except OSError:\n        return None\n    if result.returncode:\n        return None\n    try:\n        return tuple(json.loads(result.stdout.strip()))\n    except (ValueError, TypeError):\n        return None\n\n\ndef prepare_environment(project, directory="/tmp/yolo11s-grl-py312", install_packages=True,\n                        torch_index=PYPI_INDEX):\n    """Return Python 3.12 and its environment, recovering from incomplete setup."""\n    project, directory = Path(project).resolve(), Path(directory).resolve()\n    env = dict(os.environ)\n    # Kernel paths may reference Python 3.13 libraries, which must not leak here.\n    for key in ("PYTHONHOME", "PYTHONPATH", "VIRTUAL_ENV", "CONDA_PREFIX"):\n        env.pop(key, None)\n    # Kaggle\'s kernel may export an inline backend unavailable in isolated Python.\n    # All training/validation plots are saved to files and displayed by the kernel.\n    env["MPLBACKEND"] = "Agg"\n    env["UV_CACHE_DIR"] = str(directory.parent / "yolo11s-grl-uv-cache")\n    env["UV_PYTHON_INSTALL_DIR"] = str(directory.parent / "yolo11s-grl-python")\n    uv = find_uv(directory.parent / "yolo11s-grl-tools")\n    python = environment_python(directory)\n    if not (directory / "pyvenv.cfg").is_file() or python_version(python, env) != (3, 12):\n        if directory.exists():\n            # Leave existing/partial environments intact and create a fresh task-owned one.\n            directory = directory.with_name(f"{directory.name}-{uuid.uuid4().hex[:8]}")\n            python = environment_python(directory)\n        print("Creating isolated Python 3.12 environment:", directory, flush=True)\n        subprocess.run([str(uv), "venv", "--python", "3.12", str(directory)], env=env, check=True)\n        if not (directory / "pyvenv.cfg").is_file() or python_version(python, env) != (3, 12):\n            raise RuntimeError("Environment setup did not produce a working Python 3.12 interpreter")\n    if install_packages:\n        # Linux PyPI wheels include CUDA dependencies; no PyTorch-hosted index is required.\n        install_training_packages(uv, python, project, env, torch_index)\n    env["YOLO_CONFIG_DIR"] = str(project / ".ultralytics")\n    env["MPLCONFIGDIR"] = str(project / ".matplotlib")\n    print("Training Python:", python, flush=True)\n    return python, env\n', 'grl_yolo/process.py': '"""Stream subprocess errors into notebook output and preserve a complete log."""\nimport subprocess\nfrom collections import deque\nfrom pathlib import Path\n\n\ndef run_logged(arguments, cwd, env, log_file):\n    log_file = Path(log_file)\n    log_file.parent.mkdir(parents=True, exist_ok=True)\n    tail = deque(maxlen=30)\n    process = subprocess.Popen(\n        list(map(str, arguments)), cwd=cwd, env={**env, "PYTHONUNBUFFERED": "1"},\n        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,\n        encoding="utf-8", errors="replace", bufsize=1,\n    )\n    try:\n        with log_file.open("w", encoding="utf-8") as log:\n            for line in process.stdout:\n                print(line, end="", flush=True)\n                log.write(line)\n                log.flush()\n                tail.append(line)\n        code = process.wait()\n    except BaseException:\n        if process.poll() is None:\n            process.terminate()\n            try:\n                process.wait(timeout=5)\n            except subprocess.TimeoutExpired:\n                process.kill()\n                process.wait()\n        raise\n    finally:\n        process.stdout.close()\n    if code:\n        raise RuntimeError(f"Subprocess exited with code {code}. Full output: {log_file}\\n"\n                           f"Last output lines:\\n{\'\'.join(tail)}")\n    return log_file\n', 'tests/test_adaptation.py': 'import torch\n\nfrom grl_yolo.adaptation import MultiscaleDomainHeads, capture_features, reverse_gradient\nfrom grl_yolo.data import TargetImages\nfrom train import make_detector, save_detector\nfrom ultralytics import YOLO\nfrom ultralytics.cfg import get_cfg\n\n\ndef test_reverse_gradient():\n    x = torch.tensor([1.0, -2.0], requires_grad=True)\n    y = reverse_gradient(x, 0.1)\n    torch.testing.assert_close(y, x)\n    (y * torch.tensor([3.0, 4.0])).sum().backward()\n    torch.testing.assert_close(x.grad, torch.tensor([-0.3, -0.4]))\n\n\ndef test_only_feature_gradient_reverses():\n    torch.manual_seed(2)\n    heads = MultiscaleDomainHeads([32, 64, 128])\n    src = [torch.randn(2, c, 4, 4, requires_grad=True) for c in (32, 64, 128)]\n    tgt = [torch.randn(2, c, 4, 4, requires_grad=True) for c in (32, 64, 128)]\n    heads(src, tgt, -1).backward()  # ordinary gradient for reference\n    src_grads = [x.grad.clone() for x in src]\n    param_grads = [p.grad.clone() for p in heads.parameters()]\n    heads.zero_grad()\n    for x in src + tgt:\n        x.grad = None\n    heads(src, tgt, 0.1).backward()\n    for x, g in zip(src, src_grads):\n        torch.testing.assert_close(x.grad, -0.1 * g)\n    for p, g in zip(heads.parameters(), param_grads):\n        torch.testing.assert_close(p.grad, g)\n\n\ndef test_yolo11s_backward_and_checkpoint(tmp_path):\n    torch.set_num_threads(2)\n    model = make_detector({0: "damage"}, weights=None)\n    model.args = get_cfg()\n    model.train()\n    batch = {"img": torch.rand(2, 3, 64, 64), "batch_idx": torch.tensor([0, 1]),\n             "cls": torch.zeros(2, 1), "bboxes": torch.tensor([[.5, .5, .25, .25]] * 2)}\n    (loss, _), src = capture_features(model, batch)\n    _, tgt = capture_features(model, torch.rand(2, 3, 64, 64))\n    assert [f.shape[-1] for f in src] == [8, 4, 2]\n    heads = MultiscaleDomainHeads([f.shape[1] for f in src])\n    total = loss / 2 + heads(src, tgt)\n    total.backward()\n    assert torch.isfinite(total)\n    assert model.model[0].conv.weight.grad.abs().sum() > 0\n    assert all(p.grad is not None for p in heads.parameters())\n    assert not any(layer._forward_hooks for layer in model.model)\n    saved = tmp_path / "detector.pt"\n    save_detector(model, saved, model.args)\n    wrapper = YOLO(saved)\n    loaded = wrapper.model.eval()\n    with torch.no_grad():\n        prediction = loaded(torch.rand(1, 3, 64, 64))\n    assert prediction[0].shape[1] == 5  # four box coordinates + one class\n    import numpy as np\n    results = wrapper.predict(np.zeros((64, 64, 3), dtype=np.uint8), imgsz=64, device="cpu", verbose=False)\n    assert len(results) == 1 and results[0].names == {0: "damage"}\n\n\ndef test_target_ignores_labels(tmp_path):\n    import cv2\n    import numpy as np\n    images = tmp_path / "images"\n    labels = tmp_path / "labels"\n    images.mkdir()\n    labels.mkdir()\n    cv2.imwrite(str(images / "sample.jpg"), np.zeros((32, 48, 3), dtype=np.uint8))\n    (labels / "sample.txt").write_text("deliberately invalid label", encoding="utf-8")\n    dataset = TargetImages(images, 64)\n    assert dataset[0].shape == (3, 64, 64)\n\n\n', 'tests/test_domain_loss.py': '"""Analytical checks for domain BCE and domain-only backbone updates."""\nimport math\n\nimport torch\nfrom torch import nn\nfrom ultralytics.cfg import get_cfg\n\nfrom grl_yolo.adaptation import MultiscaleDomainHeads, capture_features\nfrom train import make_detector\n\n\ndef controlled_logits(scales=3):\n    """Use prescribed classifier outputs to test the loss independently of features."""\n    heads = MultiscaleDomainHeads([32] * scales)\n    heads.heads = nn.ModuleList([nn.Identity() for _ in range(scales)])\n    return heads\n\n\ndef test_domain_loss_zero_logits_is_log_two():\n    # Actual classifiers with zero parameters produce probability 0.5 everywhere.\n    heads = MultiscaleDomainHeads([32, 64, 128])\n    with torch.no_grad():\n        for parameter in heads.parameters():\n            parameter.zero_()\n    source = [torch.randn(2, c, 4, 4) for c in (32, 64, 128)]\n    target = [torch.randn(3, c, 2, 2) for c in (32, 64, 128)]\n    loss = heads(source, target)\n    torch.testing.assert_close(loss, torch.tensor(math.log(2)))\n\n\ndef test_domain_loss_balances_domains_and_averages_scales():\n    heads = controlled_logits()\n    source_logits = (0.0, 2.0, -1.0)\n    target_logits = (-3.0, 0.5, 4.0)\n    # Different batch sizes and pixel counts must not change either domain\'s weight.\n    source = [torch.full((2, 1, 8, 8), value) for value in source_logits]\n    target = [torch.full((5, 1, 2, 2), value) for value in target_logits]\n    expected = sum(\n        0.5 * (math.log1p(math.exp(-s)) + math.log1p(math.exp(t)))\n        for s, t in zip(source_logits, target_logits)\n    ) / 3\n    torch.testing.assert_close(heads(source, target), torch.tensor(expected))\n\n\ndef test_domain_loss_label_convention_and_extreme_logits():\n    heads = controlled_logits(scales=1)\n    positive = [torch.full((2, 1, 4, 4), 8.0)]\n    negative = [torch.full((2, 1, 4, 4), -8.0)]\n    # source=1 wants positive logits; target=0 wants negative logits.\n    good = heads(positive, negative)\n    wrong = heads(negative, positive)\n    torch.testing.assert_close(good, torch.tensor(math.log1p(math.exp(-8))))\n    torch.testing.assert_close(wrong, torch.tensor(math.log1p(math.exp(8))))\n    assert good < 0.001 and wrong > 8.0\n    # BCEWithLogits must remain finite even when predictions are confidently wrong.\n    extreme = heads([torch.full((2, 1, 4, 4), -1000.0)],\n                    [torch.full((2, 1, 4, 4), 1000.0)])\n    torch.testing.assert_close(extreme, torch.tensor(1000.0))\n\n\ndef test_domain_loss_alone_updates_backbone_and_grl_zero_blocks_it():\n    torch.manual_seed(7)\n    torch.set_num_threads(2)\n    model = make_detector({0: "damage"}, weights=None)\n    model.args = get_cfg()\n    model.train()\n    source_images = torch.rand(2, 3, 64, 64)\n    target_images = torch.rand(2, 3, 64, 64)\n    _, source = capture_features(model, source_images)\n    _, target = capture_features(model, target_images)\n    heads = MultiscaleDomainHeads([f.shape[1] for f in source])\n    backbone_weight = model.model[0].conv.weight\n    optimizer = torch.optim.SGD(list(model.parameters()) + list(heads.parameters()), lr=0.01)\n    before = backbone_weight.detach().clone()\n    # No detection loss or labels: the update can only come from domain loss.\n    heads(source, target, coefficient=0.1).backward()\n    assert backbone_weight.grad is not None and backbone_weight.grad.abs().sum() > 0\n    assert all(p.grad is None for p in model.model[-1].parameters())\n    optimizer.step()\n    assert not torch.equal(before, backbone_weight.detach())\n\n    optimizer.zero_grad(set_to_none=True)\n    _, source = capture_features(model, source_images)\n    _, target = capture_features(model, target_images)\n    heads(source, target, coefficient=0.0).backward()\n    torch.testing.assert_close(backbone_weight.grad, torch.zeros_like(backbone_weight))\n    assert sum(p.grad.abs().sum().item() for p in heads.parameters() if p.grad is not None) > 0\n', 'tests/test_native_trainer.py': '"""Verify default preservation, label-free augmentation and native checkpoints."""\nimport csv\nfrom copy import deepcopy\nfrom pathlib import Path\n\nimport cv2\nimport numpy as np\nimport torch\nimport yaml\nfrom ultralytics import YOLO\nfrom ultralytics.cfg import get_cfg\nfrom ultralytics.data.build import build_yolo_dataset\nfrom ultralytics.models.yolo.detect import DetectionTrainer\nfrom ultralytics.nn.tasks import DetectionModel\n\nfrom grl_yolo.config import load_training_config\nfrom grl_yolo.data import UnlabeledYOLODataset\nfrom grl_yolo.model import GRLDetectionModel\nfrom grl_yolo.trainer import GRLTrainer\nfrom train import export_checkpoints\n\n\ndef test_training_yaml_preserves_ultralytics_defaults():\n    filename = Path(__file__).resolve().parents[1] / "configs/train.yaml"\n    overrides, grl = load_training_config(filename)\n    defaults = vars(get_cfg())\n    effective = vars(get_cfg(overrides=overrides))\n    for key, value in defaults.items():\n        if key not in overrides:\n            assert effective[key] == value, key\n    assert grl["lambda"] == 0.1 and grl["layers"] == [4, 6, 10]\n    assert GRLTrainer.build_optimizer is DetectionTrainer.build_optimizer\n    assert GRLTrainer._do_train is DetectionTrainer._do_train\n    assert GRLTrainer.optimizer_step is DetectionTrainer.optimizer_step\n\n\ndef test_grl_preserves_native_detection_loss_and_adds_only_domain_term():\n    torch.set_num_threads(2)\n    model = GRLDetectionModel("yolo11s.yaml", nc=1, verbose=False)\n    model.args = get_cfg()\n    model.train()\n    source = {"img": torch.rand(2, 3, 64, 64), "batch_idx": torch.tensor([0, 1]),\n              "cls": torch.zeros(2, 1), "bboxes": torch.tensor([[.5, .5, .25, .25]] * 2)}\n    native_loss, native_items = DetectionModel.loss(model, source)\n    total, items = model({**source, "target_img": torch.rand(2, 3, 64, 64)})\n    torch.testing.assert_close(items, native_items)\n    expected = native_loss + source["img"].shape[0] * model.grl_config["domain_weight"] * model.last_domain_loss\n    torch.testing.assert_close(total, expected)\n\n\ndef make_data(root):\n    for split in ("source", "target", "val"):\n        images, labels = root / split / "images", root / split / "labels"\n        images.mkdir(parents=True)\n        labels.mkdir()\n        for index in range(4):\n            cv2.imwrite(str(images / f"{index}.jpg"), np.full((64, 64, 3), 50 + index * 40, dtype=np.uint8))\n            (labels / f"{index}.txt").write_text(\n                "0 0.5 0.5 0.25 0.25" if split != "target" else "invalid target labels",\n                encoding="utf-8")\n    filename = root / "data.yaml"\n    filename.write_text(yaml.safe_dump({"path": str(root), "train": "source/images",\n                                       "val": "val/images", "target": "target/images",\n                                       "names": ["damage"]}), encoding="utf-8")\n    return filename\n\n\ndef test_target_uses_native_augmentation_with_empty_labels(tmp_path):\n    make_data(tmp_path)\n    cfg = get_cfg(overrides={"imgsz": 64})\n    data = {"names": {0: "damage"}, "nc": 1}\n    source = build_yolo_dataset(deepcopy(cfg), str(tmp_path / "source/images"), 2, data)\n    target = UnlabeledYOLODataset(img_path=str(tmp_path / "target/images"), imgsz=64,\n                                 batch_size=2, augment=True, hyp=deepcopy(cfg), data=data)\n    def transform_signature(transform):\n        children = getattr(transform, "transforms", [])\n        return (type(transform).__name__, getattr(transform, "p", None),\n                [transform_signature(child) for child in children])\n    assert transform_signature(source.transforms) == transform_signature(target.transforms)\n    assert target[0]["img"].shape == (3, 64, 64)\n    assert target[0]["cls"].numel() == 0\n    assert all(label["cls"].shape == (0, 1) for label in target.labels)\n\n\ndef test_native_training_resume_and_plain_export(tmp_path, monkeypatch):\n    # Avoid fetching a plot font in this offline CPU verification.\n    monkeypatch.setattr("ultralytics.data.utils.check_font", lambda *args, **kwargs: None)\n    torch.set_num_threads(2)\n    data = make_data(tmp_path)\n    args = {"model": "yolo11s.yaml", "data": str(data), "epochs": 2, "imgsz": 64,\n            "batch": 2, "device": "cpu", "workers": 0, "plots": False,\n            "project": str(tmp_path / "runs"), "name": "adapted", "amp": False}\n    trainer = GRLTrainer(overrides=args)\n    # Keep two planned epochs, but emulate interruption after saving epoch one.\n    trainer.add_callback("on_model_save", lambda t: setattr(t, "stop", True))\n    trainer.train()\n    resume = trainer.wdir / "resume.pt"\n    state = torch.load(resume, weights_only=False)\n    assert state["epoch"] == 0 and state["optimizer"] is not None\n    assert isinstance(state["ema"], GRLDetectionModel)\n    assert any(key.startswith("domain_heads.") for key in state["ema"].state_dict())\n    restored = GRLTrainer(overrides={**args, "resume": str(resume)})\n    restored.train()\n    assert restored.start_epoch == 1\n    export_checkpoints(restored)\n    plain = YOLO(restored.wdir / "last_detector.pt").model\n    assert not hasattr(plain, "domain_heads")\n    with (restored.save_dir / "domain.csv").open() as stream:\n        rows = list(csv.DictReader(stream))\n    assert len(rows) == 2 and all(float(row["domain_loss"]) > 0 for row in rows)\n    # Baseline needs no target key and uses the unmodified trainer.\n    baseline_data = yaml.safe_load(data.read_text())\n    del baseline_data["target"]\n    baseline_yaml = tmp_path / "baseline.yaml"\n    baseline_yaml.write_text(yaml.safe_dump(baseline_data), encoding="utf-8")\n    baseline = DetectionTrainer(overrides={**args, "data": str(baseline_yaml), "epochs": 1, "name": "baseline"})\n    baseline.train()\n    assert (baseline.wdir / "best.pt").is_file()\n', 'tests/test_environment.py': '"""Regression checks for setup on kernels missing ensurepip or using Python 3.13."""\nimport os\nimport subprocess\nimport venv\nfrom pathlib import Path\n\nimport pytest\n\nfrom grl_yolo import environment\n\n\ndef test_reuses_real_python312_environment_without_pip(tmp_path, monkeypatch):\n    directory = tmp_path / "env"\n    # A real environment deliberately created without pip/ensurepip bootstrapping.\n    venv.EnvBuilder(with_pip=False).create(directory)\n    monkeypatch.setattr(environment, "find_uv", lambda directory: Path("fake-uv"))\n    monkeypatch.setenv("PYTHONPATH", "/kernel/python3.13/site-packages")\n    monkeypatch.setenv("MPLBACKEND", "module://matplotlib_inline.backend_inline")\n    python, env = environment.prepare_environment(tmp_path, directory, install_packages=False)\n    assert python == environment.environment_python(directory)\n    assert environment.python_version(python, env) == (3, 12)\n    assert "PYTHONPATH" not in env\n    assert env["MPLBACKEND"] == "Agg"\n    probe = subprocess.run([str(python), "-m", "pip", "--version"], env=env, capture_output=True)\n    assert probe.returncode != 0  # setup does not require pip in the environment\n\n\ndef test_recovers_partial_environment_without_deleting_it(tmp_path, monkeypatch):\n    directory = tmp_path / "partial-env"\n    directory.mkdir()\n    sentinel = directory / "keep.txt"\n    sentinel.write_text("existing files must survive")\n    commands = []\n    original_run = subprocess.run\n    monkeypatch.setattr(environment, "find_uv", lambda directory: Path("fake-uv"))\n\n    def run(arguments, **kwargs):\n        if arguments[0] == "fake-uv":\n            commands.append(arguments)\n            assert arguments[1:4] == ["venv", "--python", "3.12"]\n            # Model uv\'s result with a real, unseeded environment; no network required.\n            venv.EnvBuilder(with_pip=False).create(arguments[4])\n            return subprocess.CompletedProcess(arguments, 0)\n        return original_run(arguments, **kwargs)\n\n    monkeypatch.setattr(environment.subprocess, "run", run)\n    python, env = environment.prepare_environment(tmp_path, directory, install_packages=False)\n    assert python != environment.environment_python(directory)\n    assert environment.python_version(python, env) == (3, 12)\n    assert sentinel.read_text() == "existing files must survive"\n    assert len(commands) == 1\n\n\ndef test_incompatible_python313_is_not_reused(tmp_path, monkeypatch):\n    directory = tmp_path / "incompatible-env"\n    directory.mkdir()\n    (directory / "pyvenv.cfg").write_text("version = 3.13")\n    commands = []\n    monkeypatch.setattr(environment, "find_uv", lambda directory: Path("fake-uv"))\n    original = environment.environment_python(directory)\n    monkeypatch.setattr(environment, "python_version", lambda python, env: (3, 13) if python == original else (3, 12))\n\n    def run(arguments, **kwargs):\n        commands.append(arguments)\n        created = Path(arguments[-1])\n        created.mkdir()\n        (created / "pyvenv.cfg").write_text("version = 3.12")\n        return subprocess.CompletedProcess(arguments, 0)\n\n    monkeypatch.setattr(environment.subprocess, "run", run)\n    python, _ = environment.prepare_environment(tmp_path, directory, install_packages=False)\n    assert python != original\n    assert commands[0][1:4] == ["venv", "--python", "3.12"]\n    assert all("ensurepip" not in " ".join(command) for command in commands)\n\n\ndef test_packages_install_via_uv_with_pypi_index(tmp_path, monkeypatch):\n    directory = tmp_path / "env"\n    directory.mkdir()\n    (directory / "pyvenv.cfg").write_text("version = 3.12")\n    monkeypatch.setattr(environment, "find_uv", lambda directory: Path("fake-uv"))\n    monkeypatch.setattr(environment, "python_version", lambda python, env: (3, 12))\n    commands = []\n\n    def run(arguments, **kwargs):\n        commands.append(arguments)\n        return subprocess.CompletedProcess(arguments, 0)\n\n    monkeypatch.setattr(environment.subprocess, "run", run)\n    python, _ = environment.prepare_environment(tmp_path, directory)\n    assert len(commands) == 2\n    assert all(command[:4] == ["fake-uv", "pip", "install", "--python"] for command in commands)\n    assert commands[0][-1] == "https://pypi.org/simple"\n    assert commands[1][-4:] == ["-r", str(tmp_path / "requirements.txt"), "--index-url", environment.PYPI_INDEX]\n\n\ndef test_pytorch_index_timeout_falls_back_to_pypi(tmp_path, monkeypatch):\n    commands = []\n    custom = "https://download.pytorch.org/whl/cu124"\n\n    def run(arguments, **kwargs):\n        commands.append(arguments)\n        if arguments[-1] == custom:\n            raise subprocess.CalledProcessError(2, arguments)\n        return subprocess.CompletedProcess(arguments, 0)\n\n    monkeypatch.setattr(environment.subprocess, "run", run)\n    environment.install_training_packages("uv", "/tmp/env/bin/python", tmp_path, {}, torch_index=custom)\n    assert len(commands) == 3\n    assert commands[0][-1] == custom\n    assert commands[1][-1] == environment.PYPI_INDEX\n    assert "torch==2.5.1" in commands[1] and "torchvision==0.20.1" in commands[1]\n    assert commands[2][-1] == environment.PYPI_INDEX\n\n\ndef test_pypi_failure_stops_with_clear_error(tmp_path, monkeypatch):\n    def run(arguments, **kwargs):\n        raise subprocess.CalledProcessError(2, arguments)\n\n    monkeypatch.setattr(environment.subprocess, "run", run)\n    with pytest.raises(RuntimeError, match="Check Kaggle Internet access"):\n        environment.install_training_packages("uv", "/tmp/env/bin/python", tmp_path, {})\n\n\ndef test_invalid_interpreter_is_detected(tmp_path):\n    broken = tmp_path / "python"\n    broken.write_bytes(b"not an executable")\n    assert environment.python_version(broken, dict(os.environ)) is None\n\n\ndef test_inherited_inline_backend_does_not_break_imports(tmp_path, monkeypatch):\n    # Use the real test interpreter with installed plotting/training dependencies.\n    import sys\n    monkeypatch.setenv("MPLBACKEND", "module://matplotlib_inline.backend_inline")\n    directory = tmp_path / "env"\n    directory.mkdir()\n    (directory / "pyvenv.cfg").write_text("version = 3.12")\n    monkeypatch.setattr(environment, "find_uv", lambda directory: Path("fake-uv"))\n    monkeypatch.setattr(environment, "python_version", lambda python, env: (3, 12))\n    _, env = environment.prepare_environment(tmp_path, directory, install_packages=False)\n    result = subprocess.run([sys.executable, "-c",\n                             "import matplotlib, ultralytics; "\n                             "assert matplotlib.get_backend().lower() == \'agg\'; print(\'imports OK\')"],\n                            env=env, text=True, capture_output=True)\n    assert result.returncode == 0, result.stderr\n    assert "imports OK" in result.stdout\n', 'tests/test_process.py': 'import os\nimport sys\n\nimport pytest\n\nfrom grl_yolo.process import run_logged\n\n\ndef test_child_traceback_is_displayed_and_saved(tmp_path, capsys):\n    script = "print(\'training started\'); raise FileNotFoundError(\'source image example.jpg\')"\n    log = tmp_path / "logs/train.log"\n    with pytest.raises(RuntimeError) as error:\n        run_logged([sys.executable, "-c", script], tmp_path, dict(os.environ), log)\n    assert "FileNotFoundError: source image example.jpg" in str(error.value)\n    assert "FileNotFoundError: source image example.jpg" in log.read_text()\n    assert "training started" in capsys.readouterr().out\n\n\ndef test_success_streams_both_stdout_and_stderr(tmp_path, capsys):\n    script = "import sys; print(\'stdout message\'); print(\'stderr message\', file=sys.stderr)"\n    log = tmp_path / "success.log"\n    run_logged([sys.executable, "-c", script], tmp_path, dict(os.environ), log)\n    printed = capsys.readouterr().out\n    assert "stdout message" in printed and "stderr message" in printed\n    assert log.read_text() == printed\n'}


PROJECT_DIR.mkdir(parents=True, exist_ok=True)
for relative, content in PROJECT_FILES.items():
    path = PROJECT_DIR / relative
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(content, encoding="utf-8")
print(PROJECT_DIR)

## Optional direct file-read diagnostics
These probes print the actual file-open result; they do not stop training on a
directory metadata check. The supplied paths are passed directly to Ultralytics.

In [ ]:
sys.path.insert(0, str(PROJECT_DIR))
import importlib
import grl_yolo.kaggle as kaggle_data
importlib.reload(kaggle_data)
root = Path(DATASET_ROOT) if DATASET_ROOT else kaggle_data.discover_dataset()
for relative in PROBE_IMAGES:
    sample = root / relative
    try:
        with sample.open("rb") as stream:
            header = stream.read(16)
        print("Image read OK:", sample, "bytes checked:", len(header))
    except OSError as error:
        print("Image read FAILED:", sample, type(error).__name__, str(error))
print("Training dataset:", root)
print("File probes are diagnostic. No custom directory-access gate is applied.")

## 2. Set up the isolated Python environment
The notebook kernel may use Python 3.13. Training uses a separate **Python 3.12**
interpreter matching the pinned project dependencies. Standalone `uv` creates it
without invoking the kernel's broken/missing `ensurepip`; no kernel pip is required.
Python and package downloads need Internet. PyTorch is installed from **PyPI**
by default to avoid timeouts reaching `download.pytorch.org`. Its Linux wheel
includes CUDA dependencies; CUDA availability is checked after installation.
Training subprocesses use the `Agg` Matplotlib backend to avoid inheriting
the notebook kernel's unavailable inline backend. Saved plots display below.
The kernel does not need restarting.

In [ ]:
sys.path.insert(0, str(PROJECT_DIR))
import importlib
import grl_yolo.environment as training_environment
# Reload restored code if this notebook is imported into an existing session.
importlib.reload(training_environment)
print("Notebook kernel:", sys.version)
PYTHON, ENV = training_environment.prepare_environment(PROJECT_DIR, torch_index=TORCH_INDEX_URL)

from grl_yolo.process import run_logged
COMMAND_NUMBER = 0

def run_project(arguments):
    global COMMAND_NUMBER
    COMMAND_NUMBER += 1
    filename = "train.log" if str(arguments[0]) == "train.py" else f"command-{COMMAND_NUMBER:02d}.log"
    return run_logged([str(PYTHON), *map(str, arguments)], cwd=PROJECT_DIR, env=ENV,
                      log_file=PROJECT_DIR / "logs" / filename)

runtime = subprocess.run(
    [str(PYTHON), "-c", "import sys; print('Training Python:', sys.version, flush=True); "
     "import torch, ultralytics; print(torch.__version__, ultralytics.__version__, flush=True); "
     "print('CUDA runtime:', torch.version.cuda, 'CUDA available:', torch.cuda.is_available(), flush=True); "
     "assert torch.cuda.is_available(), 'Enable a GPU in Kaggle settings'; "
     "print(torch.cuda.get_device_name(0), flush=True)"],
    cwd=PROJECT_DIR, env=ENV, text=True, capture_output=True,
)
print(runtime.stdout)
if runtime.stderr:
    print(runtime.stderr)
if runtime.returncode:
    raise RuntimeError("Training runtime check failed. The underlying error is printed above.")
if RUN_TESTS:
    subprocess.run([str(PYTHON), "-m", "pytest", "-q"], cwd=PROJECT_DIR,
                   env={**ENV, "OMP_NUM_THREADS": "2", "MKL_NUM_THREADS": "2"}, check=True)

## 3. Write dataset YAML with the exact input paths
Source and target images are read directly from the supplied Kaggle locations.
No directory discovery, symlink staging or additional access checks are used here.
Native Ultralytics loading validates the source/validation data. Target training
uses the same image-loading pipeline without reading target annotations.
A read-only label-cache warning can be ignored; source images do not need copying.

In [ ]:
SOURCE_IMAGES = root / "source/images"
TARGET_IMAGES = root / TARGET_SPLIT / "images"
VALIDATION_IMAGES = root / VALIDATION_SPLIT / "images"
TEST_IMAGES = root / TEST_SPLIT / "images" if TEST_SPLIT else None
has_test = TEST_IMAGES is not None
dataset = {"path": str(root), "train": str(SOURCE_IMAGES), "target": str(TARGET_IMAGES),
           "val": str(VALIDATION_IMAGES), "names": NAMES}
if has_test:
    dataset["test"] = str(TEST_IMAGES)
DATA_CONFIG = PROJECT_DIR / "configs/kaggle_data.yaml"
DATA_CONFIG.write_text(json.dumps(dataset, indent=2), encoding="utf-8")
print(DATA_CONFIG.read_text())

## 4. Load the architecture YAML and edit training settings
Omitted `train` keys inherit Ultralytics defaults. Uncomment only settings you
want to override. `device: 0` selects one GPU; it does not change optimizer or
augmentation defaults. Edit `configs/yolo11s-grl.yaml` to change backbone,
detection head or GRL branch settings. The training YAML selects that file and
initializes matching weights from `yolo11s.pt`, downloading them if needed.
`project` and `name` are omitted so Ultralytics chooses the default run folder.

In [ ]:
TRAINING_YAML = """
train:
  model: yolo11s-grl.yaml
  pretrained: yolo11s.pt
  data: kaggle_data.yaml
  device: 0
  # batch: 4
  # workers: 2
  # epochs: 100
  # imgsz: 640
  # optimizer: auto
  # lr0: 0.01
  # mosaic: 1.0
  # mixup: 0.0
  # close_mosaic: 10

# Optional GRL experiment overrides (defaults are in the model YAML):
# grl:
#   lambda: 0.2
#   domain_weight: 0.5
"""
TRAIN_CONFIG = PROJECT_DIR / "configs/kaggle_train.yaml"
from textwrap import dedent
TRAIN_CONFIG.write_text(dedent(TRAINING_YAML), encoding="utf-8")
print(TRAIN_CONFIG.read_text())

## 5. Native training, validation and best-model selection
Subprocess stdout/stderr are displayed live and saved to `logs/train.log`.
On failure, the exception includes the final training error and the log path.
Outputs include native `results.csv`, `args.yaml`, validation plots and GRL
`domain.csv`. `weights/resume.pt` preserves domain heads, EMA and optimizer
for an interrupted run. Resume uses the original training budget and settings;
completed runs should initialize a new run from a detector checkpoint instead.
Previous standalone-loop `adaptation.pt` checkpoints are incompatible with this trainer.

In [ ]:
report = PROJECT_DIR / "notebook_run.json"
command = ["train.py", "--config", TRAIN_CONFIG, "--report", report]
if RESUME:
    command += ["--resume", RESUME]
run_project(command)
RUN_DIR = Path(json.loads(report.read_text())["run_dir"])
print("Completed run:", RUN_DIR)
print("Resume:", RUN_DIR / "weights/resume.pt")

## 6. Optional baseline with the same native settings
Enable `RUN_BASELINE` in the first cell for a second run without adaptation.

In [ ]:
BASELINE_DIR = None
if RUN_BASELINE:
    baseline_report = PROJECT_DIR / "notebook_baseline.json"
    run_project(["train.py", "--config", TRAIN_CONFIG, "--baseline", "--report", baseline_report])
    BASELINE_DIR = Path(json.loads(baseline_report.read_text())["run_dir"])
else:
    print("Baseline skipped.")

## 7. Inspect native training results and domain loss

In [ ]:
from IPython.display import Image, display
native_plot = RUN_DIR / "results.png"
if native_plot.exists():
    display(Image(filename=str(native_plot)))
domain_csv = RUN_DIR / "domain.csv"
if domain_csv.exists():
    plotting = '''
import csv
import sys
from pathlib import Path
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
run = Path(sys.argv[1])
with (run / "domain.csv").open() as f:
    rows = list(csv.DictReader(f))
epochs = [int(r["epoch"]) for r in rows]
fig, axes = plt.subplots(1, 2, figsize=(10, 4))
for ax, key in zip(axes, ("domain_loss", "grl_lambda")):
    ax.plot(epochs, [float(r[key]) for r in rows])
    ax.set_xlabel("Epoch")
    ax.set_ylabel(key)
fig.tight_layout()
fig.savefig(run / "domain.png", dpi=150)
plt.close(fig)
'''
    run_project(["-c", plotting, RUN_DIR])
    display(Image(filename=str(RUN_DIR / "domain.png")))
print("Effective settings:", RUN_DIR / "args.yaml")

## 8. Evaluate the held-out target test set
Best detector weights are selected using validation fitness. Final test
results are separate from training and validation. Exported detector checkpoints
contain no GRL modules and load with standard Ultralytics.

In [ ]:
evaluation = '''
import json
import sys
from pathlib import Path
from ultralytics import YOLO
weights, data, report = sys.argv[1:]
metrics = YOLO(weights).val(data=data, split="test", device=0)
result = {k: float(v) for k, v in metrics.results_dict.items()}
Path(metrics.save_dir, "metrics.json").write_text(json.dumps(result, indent=2))
Path(report).write_text(json.dumps({"save_dir": str(Path(metrics.save_dir).resolve())}))
print(json.dumps(result, indent=2))
'''
EVALUATION_DIRS = []
if has_test:
    models = [("adapted", RUN_DIR)] + ([("baseline", BASELINE_DIR)] if BASELINE_DIR else [])
    for label, folder in models:
        evaluation_report = PROJECT_DIR / f"evaluation_{label}.json"
        run_project(["-c", evaluation, folder / "weights/best_detector.pt", DATA_CONFIG, evaluation_report])
        EVALUATION_DIRS.append(Path(json.loads(evaluation_report.read_text())["save_dir"]))
else:
    print("Final test split disabled; final evaluation skipped.")

## 9. Preview predictions and package results
Download `/kaggle/working/yolo11s-grl-results.zip` from notebook outputs.
The archive excludes dataset symlinks, caches and the Python environment.

In [ ]:
preview_images = TEST_IMAGES if has_test else VALIDATION_IMAGES
preview = next(p for p in sorted(preview_images.rglob("*"))
               if p.is_file() and p.suffix.lower() in {".jpg", ".jpeg", ".png"})
prediction = '''
import sys
import json
from pathlib import Path
from ultralytics import YOLO
weights, source, report = sys.argv[1:]
results = YOLO(weights).predict(source=source, device=0, save=True)
Path(report).write_text(json.dumps({"save_dir": str(Path(results[0].save_dir).resolve())}))
'''
prediction_report = PROJECT_DIR / "prediction_run.json"
run_project(["-c", prediction, RUN_DIR / "weights/best_detector.pt", preview, prediction_report])
PREDICTION_DIR = Path(json.loads(prediction_report.read_text())["save_dir"])
for file in sorted(PREDICTION_DIR.glob("*"))[:1]:
    display(Image(filename=str(file)))
archive = Path("/kaggle/working/yolo11s-grl-results.zip")
with zipfile.ZipFile(archive, "w", compression=zipfile.ZIP_DEFLATED) as z:
    for relative in PROJECT_FILES:
        z.write(PROJECT_DIR / relative, arcname=f"project/{relative}")
    for config in (DATA_CONFIG, TRAIN_CONFIG):
        z.write(config, arcname=f"project/configs/{config.name}")
    run_folders = [("train", RUN_DIR), ("predict", PREDICTION_DIR), ("logs", PROJECT_DIR / "logs")]
    if BASELINE_DIR:
        run_folders.append(("baseline", BASELINE_DIR))
    run_folders += [(f"val-{i}", folder) for i, folder in enumerate(EVALUATION_DIRS)]
    for label, folder in run_folders:
        for file in folder.rglob("*"):
            if file.is_file() and not file.is_symlink():
                z.write(file, arcname=f"results/{label}/{file.relative_to(folder).as_posix()}")
print("Download:", archive)